# What 0.5 Added

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/structured-world/coordinode-python/blob/main/demo/notebooks/04_whats_new_in_0_5.ipynb)

The client surface that arrived with CoordiNode 0.5, exercised end to end:

| Feature | What it is for |
|---------|----------------|
| `create_nodes_batch` | Create many nodes in one atomic write instead of a loop of round trips |
| `element_id` | A stable identifier that survives restarts and replication |
| `schema_revision` | Tells you when a label's shape last changed, so caches know to refresh |
| `write_concern` | Chooses how durable an acknowledgement is, from fire-and-forget to majority |
| `read_concern` / `read_preference` | Chooses how fresh a read is, and which replica answers it |
| `at_timestamp` | Reads the database as it was at a point in time |

> **Needs a server.** These are distribution and durability features, so they
> exist on the client that talks to a CoordiNode server. Set `COORDINODE_ADDR`
> before running. The embedded engine has no Raft and no replicas, so the
> cells below stop with an explanation rather than pretending.


## Install dependencies


In [ ]:
import importlib.util, inspect, os, shutil, subprocess, sys

# (distribution on PyPI, module to import it by).
pkgs = [
    ("coordinode", "coordinode"),
    ("nest_asyncio", "nest_asyncio"),
]

# Install only what is missing. A checkout mounted into the image, or an
# editable install, already provides these; pulling them from PyPI there
# would shadow the very code the notebook is meant to exercise.
#
# find_spec raises rather than returning None when a dotted name's parent
# package is absent, which is exactly the fresh environment this cell exists
# for: unguarded, it would abort before installing anything.
def _missing(module: str) -> bool:
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:
        return True


def _install(dists: list[str], upgrade: bool = False) -> None:
    """Install into this interpreter with whatever installer it has.

    Colab ships pip. A uv-managed venv deliberately does not, and there
    `python -m pip` dies with "No module named pip" before the notebook
    reaches its first query, so fall back to uv targeting this same
    interpreter rather than uv's own default environment.

    The finder caches a directory listing taken before the install, so a
    package that has just appeared on disk is invisible until the caches are
    dropped. Doing it here means every caller sees what it installed.
    """
    flags = ["-q", "-U"] if upgrade else ["-q"]
    if not _missing("pip"):
        cmd = [sys.executable, "-m", "pip", "install", *flags, *dists]
    elif shutil.which("uv"):
        cmd = ["uv", "pip", "install", *flags, "--python", sys.executable, *dists]
    else:
        raise RuntimeError(
            f"Neither pip nor uv is available to install: {', '.join(dists)}"
        )
    subprocess.run(cmd, check=True, timeout=300)
    importlib.invalidate_caches()


def _has_0_5_surface() -> bool:
    """Whether the installed `coordinode` has the API this notebook calls.

    Every cell below is about something 0.5 added, so "the module imports" is
    the wrong question: a 0.4 install answers it happily and then fails four
    cells later with an AttributeError about `create_nodes_batch`, which reads
    like a broken notebook rather than an old package. Two probes cover the
    whole set, since these all shipped together: the batch method, and the
    consistency arguments on `cypher`.

    Probing the surface rather than comparing `__version__` is deliberate. An
    editable install without hatch-vcs reports 0.0.0, so a version floor would
    reject exactly the checkout this notebook is meant to exercise.
    """
    try:
        from coordinode import CoordinodeClient
    except ImportError:
        return False
    if not hasattr(CoordinodeClient, "create_nodes_batch"):
        return False
    params = inspect.signature(CoordinodeClient.cypher).parameters
    return {"read_concern", "write_concern", "at_timestamp"}.issubset(params)


missing = [dist for dist, mod in pkgs if _missing(mod)]
if missing:
    _install(missing)

if not _has_0_5_surface():
    print("Installed `coordinode` predates 0.5, upgrading it.")
    _install(["coordinode"], upgrade=True)
    # The old module object stays in sys.modules and would keep answering
    # imports for the rest of the session, so drop it and re-probe.
    for name in [m for m in sys.modules if m == "coordinode" or m.startswith("coordinode.")]:
        del sys.modules[name]
    if not _has_0_5_surface():
        raise RuntimeError(
            "`coordinode` still lacks the 0.5 client surface after upgrading. "
            "If this environment mounts a checkout or uses an editable install, "
            "that source is older than 0.5: update it, or install the released "
            "package instead."
        )

import nest_asyncio

nest_asyncio.apply()

print("Ready")


## Connect

Every cell below guards on `client`, so running the notebook without a server
produces one explanation instead of a cascade of failures.


In [ ]:
import os, time, uuid

COORDINODE_ADDR = os.environ.get("COORDINODE_ADDR")
client = None

if COORDINODE_ADDR:
    from coordinode import CoordinodeClient

    client = CoordinodeClient(COORDINODE_ADDR)
    if not client.health():
        client.close()
        raise RuntimeError(f"Health check failed for {COORDINODE_ADDR}")
    print(f"Connected to {COORDINODE_ADDR}")
else:
    print(
        "COORDINODE_ADDR is not set, so this notebook has nothing to talk to.\n"
        "Start the demo stack (see demo/README.md) or point at your own server:\n"
        '  os.environ["COORDINODE_ADDR"] = "localhost:37080"'
    )

# Tag every node this run creates, so the cleanup at the end removes exactly
# what was added and nothing a sibling notebook left behind.
DEMO_TAG = f"whats_new_{uuid.uuid4().hex[:8]}"
print(f"DEMO_TAG: {DEMO_TAG}")


## Atomic bulk insert

`create_nodes_batch` sends the whole set as one write. A loop over
`create_node` costs a round trip each and, more importantly, can leave half
the batch committed if the process dies midway; the batch either lands
completely or not at all.


In [ ]:
if client:
    people = [
        (["Engineer"], {"name": name, "team": team, "tag": DEMO_TAG})
        for name, team in [
            ("Ada", "storage"),
            ("Grace", "storage"),
            ("Linus", "kernel"),
            ("Barbara", "query"),
            ("Edsger", "query"),
        ]
    ]
    created = client.create_nodes_batch(people)
    print(f"Created {len(created)} nodes in one write")
    for n in created[:3]:
        print(f"  {n.properties['name']:<8} id={n.id}  element_id={n.element_id}")


## Stable identity: `element_id`

`id` is a numeric handle kept for Neo4j v4 driver compatibility. `element_id`
is the canonical one: stable across restarts, schema changes and replication,
and it is what application code should store when it needs to point at a node
later. On an edge it names the two endpoints in canonical order, because an
edge here is a typed property bag between two nodes rather than an entity with
its own identity.


In [ ]:
if client:
    ada, grace = created[0], created[1]
    edge = client.create_edge("PAIRS_WITH", ada.id, grace.id, {"since": 2026})
    print(f"node  element_id: {ada.element_id}")
    print(f"edge  element_id: {edge.element_id}   ({edge.type})")

    # The handle survives a re-read: fetch the node again and compare.
    again = client.get_node(ada.id)
    print(f"re-read matches:   {again.element_id == ada.element_id}")


## Schema revision

Every label and edge type carries a revision that moves when its shape
changes. A client that caches a schema compares revisions instead of
re-fetching and re-parsing the whole thing.


In [ ]:
if client:
    labels = {l.name: l for l in client.get_labels()}
    engineer = labels.get("Engineer")
    if engineer:
        print(f"Engineer schema_revision: {engineer.schema_revision}")
    for et in client.get_edge_types():
        if et.name == "PAIRS_WITH":
            print(f"PAIRS_WITH schema_revision: {et.schema_revision}")


## Write concerns: how durable is "done"

A write concern is the answer to "acknowledged by whom". They rise in
durability:

| Concern | Acknowledged when |
|---------|-------------------|
| `w0` | The server accepted the request; nothing is guaranteed |
| `memory` | Applied in memory, before Raft |
| `cache` | Cached, still before Raft |
| `w1` | The leader has it durably (default) |
| `majority` | A majority of the cluster has it |

`memory` and `cache` acknowledge before the write reaches Raft, so a leader
crash before the background drain loses them. They are for data you can
afford to lose, not for a speed-up on data you cannot.


In [ ]:
if client:
    for concern in ("w0", "memory", "cache", "w1", "majority"):
        started = time.perf_counter()
        client.cypher(
            "CREATE (:Sample {tag: $tag, concern: $concern})",
            params={"tag": DEMO_TAG, "concern": concern},
            write_concern=concern,
        )
        elapsed_ms = (time.perf_counter() - started) * 1000
        print(f"  {concern:<9} acknowledged in {elapsed_ms:6.2f} ms")


## Read concerns and read preference

A read concern says how fresh the answer must be; a read preference says which
node may answer. On the single-node demo stack every combination returns the
same rows, which is the point: the same code runs unchanged against a cluster,
where the choice starts to matter.


In [ ]:
if client:
    for concern in ("local", "majority", "linearizable", "snapshot"):
        rows = client.cypher(
            "MATCH (n:Sample {tag: $tag}) RETURN count(n) AS n",
            params={"tag": DEMO_TAG},
            read_concern=concern,
        )
        print(f"  read_concern={concern:<13} -> {rows[0]['n']} rows")

    rows = client.cypher(
        "MATCH (n:Sample {tag: $tag}) RETURN count(n) AS n",
        params={"tag": DEMO_TAG},
        read_preference="primary_preferred",
    )
    print(f"  read_preference=primary_preferred -> {rows[0]['n']} rows")


## Time travel: `at_timestamp`

`at_timestamp` pins a read to a version of the database instead of waiting for
one. Two things it is easy to get wrong:

- The value is **microseconds since the Unix epoch**, so `int(time.time() * 1_000_000)`
  is now. Pass a millisecond value and the read silently lands in the far past;
  pass one scaled too high and it lands in the present and looks like a no-op.
- It requires `read_concern="snapshot"`. Any other level is refused with
  `FAILED_PRECONDITION`, because reading at a pinned version is exactly what a
  snapshot read is.

The timestamp also has to fall inside the MVCC retention window. Ask for
something older than the server still keeps and it answers `UNAVAILABLE` rather
than quietly returning the oldest thing it has.


In [ ]:
def now_micros() -> int:
    """The timestamp CoordiNode reads at: microseconds since the Unix epoch."""
    return int(time.time() * 1_000_000)


if client:
    client.cypher(
        "CREATE (:Era {tag: $tag, name: 'before'})",
        params={"tag": DEMO_TAG},
        write_concern="majority",
    )
    time.sleep(1.5)
    mark = now_micros()
    time.sleep(1.5)
    client.cypher(
        "CREATE (:Era {tag: $tag, name: 'after'})",
        params={"tag": DEMO_TAG},
        write_concern="majority",
    )

    query = "MATCH (n:Era {tag: $tag}) RETURN n.name AS name ORDER BY name"
    now = [r["name"] for r in client.cypher(query, params={"tag": DEMO_TAG})]
    past = [
        r["name"]
        for r in client.cypher(
            query,
            params={"tag": DEMO_TAG},
            at_timestamp=mark,
            read_concern="snapshot",
        )
    ]
    print(f"  now         : {now}")
    print(f"  at the mark : {past}")
    # Check both sides. `"after" not in past` alone also passes when the
    # past read came back empty, which is the failure this cell exists to
    # catch: a snapshot read that returns nothing looks exactly like a
    # snapshot read that correctly hides one node.
    problems = []
    if "before" not in now or "after" not in now:
        problems.append(f"current view should hold both eras, got {now}")
    if "before" not in past:
        problems.append(f"the earlier write must be visible at the mark, got {past}")
    if "after" in past:
        problems.append(f"the later write must be invisible at the mark, got {past}")
    if problems:
        raise RuntimeError("time travel returned the wrong view: "
                           + "; ".join(problems))
    print("  the mark sees only what existed then")


## Clean up


In [ ]:
if client:
    for label in ("Engineer", "Sample", "Era"):
        client.cypher(
            f"MATCH (n:{label} {{tag: $tag}}) DETACH DELETE n",
            params={"tag": DEMO_TAG},
        )
    client.close()
    print(f"Removed everything tagged {DEMO_TAG}")
